# CS156 Assignment #1

## 1. What am I trying to do? 

I use an OpenClaw personal assistant via Telegram. I've found I mostly use it to set reminders. That's a problem because 1) it costs me money since I'm paying for API tokens, 2) those asks are very simple and could be done by a local model, and 3) I'd rather not send my personal messages to a cloud provider. In its lifetime, 238 out of 558 messages I've sent had the word 'remind' or 'reminder' on them.

To fix this, I build a logistic classifier that decides whether a message is a reminder-related ask, so I can route them accordingly to a local model when they are or to a cloud provider when they're not. I compare it with two approaches: a regex for 'remind' in incoming messages, and a naive Bayes classifier. I use a larger decision model to label the messages and then train a small, fast model on those labels using a TF-IDF table and logistic regression. 

## 2. Pipeline Diagram

```
 REMINDER CLASSIFIER PIPELINE

 ┌─────────────────────┐   ┌─────────────────────┐   ┌─────────────────────┐
 │ 1) Telegram export  │   │ 2) Parse & clean    │   │ 3) Label locally    │
 ├─────────────────────┤   ├─────────────────────┤   ├─────────────────────┤
 │ HTML of my chat     │──▶│ drop setup days,    │──▶│ Decider 27B via     │
 │ with the assistant  │   │ fillers, duplicates │   │ Ollama, on device   │
 │ 679 messages        │   │ 558 messages        │   │ 257 rem · 301 other │
 └─────────────────────┘   └─────────────────────┘   └──────────┬──────────┘
            ┌───────────────────────────────────────────────────┘
            ▼
 ┌─────────────────────┐   ┌─────────────────────┐   ┌─────────────────────┐
 │ 4) Redact & save    │   │ 5) Stratified split │   │ 6) TF-IDF table     │
 ├─────────────────────┤   ├─────────────────────┤   ├─────────────────────┤
 │ PII → <redacted>    │──▶│ 80 / 20, same class │──▶│ a column per word,  │
 │ one row per message │   │ mix in both sets    │   │ fit on train only   │
 │ messages.csv        │   │ 446 train, 112 test │   │ 1,883 words         │
 └─────────────────────┘   └──────────┬──────────┘   └──────────┬──────────┘
            ┌─────────────────────────┘                         │
            │ raw text                ┌─────────────────────────┤
            ▼                         ▼                         ▼
 ┌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┐   ┌─────────────────────┐   ╔═════════════════════╗
 ╎ Regex               ╎   │ Naive Bayes         │   ║ Logistic regression ║
 ├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤   ├─────────────────────┤   ╟─────────────────────╢
 ╎ \bremind\w*         ╎   │ on TF-IDF           │   ║ on TF-IDF           ║
 ╎ no training         ╎   │                     │   ║                     ║
 ╎ baseline            ╎   │ comparison          │   ║ my model            ║
 └╌╌╌╌╌╌╌╌╌╌┬╌╌╌╌╌╌╌╌╌╌┘   └──────────┬──────────┘   ╚══════════╤══════════╝
            └─────────────────────────┼─────────────────────────┘
                                      ▼
 ┌─────────────────────────────────────────────────────────────────────────┐
 │ 7) Evaluate & compare on the 112 test messages                          │
 ├─────────────────────────────────────────────────────────────────────────┤
 │ accuracy · precision · recall · false positives / negatives             │
 │ confusion matrix · mistakes · learning curves (train set)               │
 └─────────────────────────────────────────────────────────────────────────┘
```

Note: I was trying to make my diagram on https://www.mockdown.design but the software was tough to use, so I instead asked Claude to do the diagram I had in mind on that style in the notebook directly and it turned out well, so I decided to keep it. 


## 3. The Data


I exported my messages from Telegram, parsed them, stripped them down by dropping the first 2 days (they were mostly setup), filler word replies like "yes" and any duplicates by checking hashes. 

I then used one of the "System 1" models (Perplexity Decider v1.1 27B via Ollama) to locally classify each message as reminder-related (257) or other (301). I used its labels as the ground-truth. The naive reminder regex agrees with them on 95.2% of messages. Most of its 27 disagreements (23) are reminders that don't contain the word 'remind' but do have the intent after I reviewed them. This is why I trusted these labels as the ground-truth. I redacted names and other PII before using the data.

I split 80% of the messages to train and 20% to test, stratified by label so both sets keep the same class proportions. This left me with 446 training and 112 test messages. I wrote my thought process in the implementation notebook.

## 4. Repository

https://github.com/SomeoneElseSt/cs-156/tree/master/assignment-1

## 5. Novel Equations

I did not work with any novel equations we didn't see in class. We saw TF-IDF and Naive Bayes in Session 5 and logistic regression in Session 6. I thought to include the equations that sklearn uses to implement the methods I used, but that wouldn't be adding anything substantive not already covered. 

## 6. Novel Algorithms/Metrics

Now, you could say using Perplexity's model technically was using a novel algorithm, but I couldn't explain it even if I wanted to because they didn't release how it was trained. The announcement post only mentions that they did reinforcement learning on Qwen3.8-27B. As far as this write up goes, the algorithm was POSTing to Ollama locally.

## 7. Model Learning

I do not have any loss graphs given training consisted of simply fitting the data. However, I did make a graph to compare the loss after training on different amounts of data. I was curious whether it was picking out a single word (e.g. variations of remind) early on and then stopped learning. It doesn't! The loss curves fall smoothly as more data is provided in the logistic regression approach as shown below. 

![Learning curve](figures/learning_curve_logistic.png)

I also did the same curve but for Naive Bayes.

![Learning curve for naive Bayes](figures/learning_curve_naive_bayes.png)

The average slope on naive Bayes is less aggressive. Here, it does learn rather early and then doesn't seem to gain as much from more data. I would guess the reason why is because it overfocuses on the individual role of words, identifies a few early on that contribute greatly and then doesn't go beyond that. Though, this might be because there is some hidden floor for both methods that naive Bayes reaches earlier, as seen by a lower initial loss. 

These plots would make me feel more confident in scaling a logistic regressor but I'd try more data on both. 

## 8. Final Model Performance

These are my model's stats:

| Model | Accuracy | Precision | Recall |
|---|---|---|---|
| Logistic regression | 96.4% | 96.2% | 96.2% |

And this is its confusion matrix on the test set:

![Confusion matrix for the test set](figures/confusion_matrix.png)

Of 112 messages in the test set, it classified 50/52 as reminder-related correctly and 58/60 as other correctly. 


## 9. Comparison

These were the results for the models I tried on the same 112 test messages:

| Model | Accuracy | Precision | Recall | False positives | False negatives |
|---|---|---|---|---|---|
| Logistic regression | 96.4% | 96.2% | 96.2% | 2 | 2 |
| Naive Bayes | 93.8% | 89.5% | 98.1% | 6 | 1 |
| Regex (`\bremind\w*`) | 99.1% | 100.0% | 98.1% | 0 | 1 |

Naive Bayes had many more false positives and worse precision. When inspecting some of its false positives and false negatives, some blatant examples stood out: 

`true: reminder | predicted: other    | Yeah add it to the reminder`

`true: other    | predicted: reminder | Find me the arxiv pdf for this paper please`

Logistic regression wasn't perfect but it didn't have as many issues:

`true: other    | predicted: reminder | At 11:30`

`true: other    | predicted: reminder | to cold email <redacted>`

At least it didn't think me asking for an ArXiv link was a reminder ask, so I'm happy.

The regex seems like it did really well but that's only because it got lucky. Since it's deterministic, I know 26 out of its 27 disagreements must've gone into the training set. Its original accuracy was 95.2%. Even if that's not bad, it would still miss asks without the word reminder in them, so it's not practical.

## 10. Discussion

I'm a bit scared the model might have overfit, but I actually had an easy way to test this. As I went about my day after making the model, I tried some messages I'd send my assistant asking for reminders, and the model got most of them right!

I think this wasn't a tough classification problem because I'm fairly predictable. If you look at the top 30 words pushing at each class, I found it interesting that 'remind', 'me' and 'to', which I usually write together, were the strongest signals towards reminder. The model can't see word order, but it picked up on the words I most often use in my reminder asks. 

Though, the most surprising finding was that changing 'you' to 'u' in the sentence 'Can you/u cancel that reminder' switched the class from other to reminder. It's clear why in the graphs, 'you' is really downweighted (I believe because I mainly address the assistant as you when I have a request, like 'can you do XYZ', so it learnt to associate that). It makes me think about the fragility of these systems.

On the bright side, I now have a functional model for my use case that can run even in the tightly resource constrained virtual machine my assistant lives on. It was also fun to learn you can distill bigger models into much simpler and interpretable math functions. I am now curious as to the extent to which this generalizes for distilling other labelling tasks. 


## 11. References

- Artificial Analysis, "Artificial Analysis Decision Index": https://artificialanalysis.ai/articles/artificial-analysis-decision-index
- Perplexity, "pplx-decider-v1.1-27b, our updated open weights multimodal decision model, is now available" (October 6, 2026): https://community.perplexity.ai/t/pplx-decider-v1-1-27b-our-updated-open-weights-multimodal-decision-model-is-now-available/6286
- Perplexity, "pplx-decider-v1.1-27b" model card (Hugging Face): https://huggingface.co/perplexity-ai/pplx-decider-v1.1-27b
- scikit-learn, logistic regression example: https://scikit-learn.org/stable/auto_examples/linear_model/plot_logistic_multinomial.html
- scikit-learn, `TfidfVectorizer`: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
- scikit-learn, `LogisticRegression`: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
- scikit-learn, `MultinomialNB`: https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html
- scikit-learn, `train_test_split`: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
- scikit-learn, `learning_curve`: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.learning_curve.html
- Watson, P. (2026). *CS156 Session 5 pre-class work: SMS spam classification with TF-IDF and naive Bayes* [Jupyter notebook]. Minerva University.
- Watson, P. (2026). *CS156 Session 6* [Class session]. Minerva University.
- Kharkar, R. (2019). *Maximum Likelihood for Logistic Regression*. Session 6 resource.
